# Fragility and the global D1

This notebook measures how much of the world's poorest decile lives in countries the World Bank classifies as fragile or conflict-affected.

Like the service-coverage notebooks, this only counts **where people live**, so it carries no assumption about who within a country is poor.

## Source

`data/C2-FCSList-FY06toFY26.pdf`, the World Bank's historical list, parsed by `parse_fcs_list.py`.

## A break in the series

The list is not one consistent series. It was built three different ways, so **only the FCS era is comparable over time**:

| Fiscal years | Methodology | Comparable? |
| --- | --- | --- |
| FY06-FY08 | LICUS, driven by CPIA ratings | No |
| FY09-FY19 | Harmonized list of fragile situations | No |
| FY20-FY26 | Current FCS methodology | Yes |

A fiscal year list is published partway through the previous calendar year, so FY26 is mapped to calendar 2025.

In [4]:
from pathlib import Path
import sys
import importlib
import altair as alt
import pandas as pd

NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / 'parse_fcs_list.py').exists() else Path.cwd() / 'dev_indicators'
sys.path.insert(0, str(NOTEBOOK_DIR))
import parse_fcs_list
import indicator_analysis_helpers
importlib.reload(parse_fcs_list)
importlib.reload(indicator_analysis_helpers)
from parse_fcs_list import parse_fcs
from indicator_analysis_helpers import load_d1_shares

PDF = NOTEBOOK_DIR.parent / 'data' / 'C2-FCSList-FY06toFY26.pdf'
fcs = parse_fcs(PDF)
d1 = load_d1_shares()

rows = []
for (fiscal_year, methodology), group in fcs.groupby(['fiscal_year', 'methodology']):
    year = fiscal_year - 1
    panel = d1[d1['year'] == year]
    if panel.empty:
        continue
    listed = panel[panel['code'].isin(set(group['code']))]
    d1_share = 100 * listed['d1_population_share'].sum()
    world_share = 100 * listed['country_population'].sum() / panel['country_population'].sum()
    rows.append({
        'year': year,
        'fiscal_year': fiscal_year,
        'methodology': methodology,
        'd1_share': d1_share,
        'world_share': world_share,
        'gap_ratio': d1_share / world_share,
        'countries_listed': group['code'].nunique(),
        'countries_matched': listed['code'].nunique(),
    })
concentration = pd.DataFrame(rows).sort_values('year').reset_index(drop=True)
unmatched = concentration.query('countries_listed != countries_matched')
print(f"Years parsed: {len(concentration)} | list entries never matched to D1 data: {len(unmatched)}")
display(concentration.round(2))

Years parsed: 21 | list entries never matched to D1 data: 3


,year,fiscal_year,methodology,d1_share,world_share,gap_ratio,countries_listed,countries_matched
0,2005,2006,LICUS,26.26,7.63,3.44,35,35
1,2006,2007,LICUS,25.97,7.66,3.39,35,35
2,2007,2008,LICUS,20.00,5.43,3.68,34,34
3,2008,2009,Fragile states,20.21,6.13,3.30,35,35
4,2009,2010,Harmonized fragile situations,18.15,6.11,2.97,37,36
5,2010,2011,Harmonized fragile situations,17.37,6.19,2.81,33,32
6,2011,2012,Harmonized fragile situations,18.16,6.16,2.95,33,32
7,2012,2013,Harmonized fragile situations,18.72,6.73,2.78,35,35
8,2013,2014,Harmonized fragile situations,23.69,7.08,3.34,36,36
9,2014,2015,Harmonized fragile situations,22.71,6.48,3.51,33,33


## Where the poorest decile lives

The orange line is the share of the global D1 living in listed countries; the blue line is the same share for the world population. The dashed rule marks the FY20 methodology change, so only the segment to its right is a like-for-like trend.

In [7]:
long = concentration.melt(
    id_vars=['year', 'methodology'],
    value_vars=['d1_share', 'world_share'],
    var_name='group',
    value_name='share',
).replace({'d1_share': 'Poorest global decile', 'world_share': 'World population'})

lines = (
    alt.Chart(long)
    .mark_line(strokeWidth=3)
    .encode(
        x=alt.X('year:Q', title='Year', axis=alt.Axis(format='d')),
        y=alt.Y('share:Q', title='Living in fragile or conflict-affected countries (%)'),
        color=alt.Color(
            'group:N',
            title=None,
            scale=alt.Scale(
                domain=['Poorest global decile', 'World population'],
                range=['#D55E00', '#0072B2'],
            ),
        ),
        detail='methodology:N',
        tooltip=[
            alt.Tooltip('year:O', title='Year'),
            alt.Tooltip('group:N', title='Group'),
            alt.Tooltip('share:Q', title='Share (%)', format='.1f'),
            alt.Tooltip('methodology:N', title='Methodology'),
        ],
    )
)
methodology_break = (
    alt.Chart(pd.DataFrame({'year': [2018.5]}))
    .mark_rule(strokeDash=[6, 4], color='#666666')
    .encode(x='year:Q')
)
fcs_chart = (lines + methodology_break).properties(
    width=760,
    height=380,
    title='Share of the poorest global decile living in fragile or conflict-affected countries',
)
fcs_chart

alt.LayerChart(...)

## Comparable trend: the FCS era

In [6]:
era = concentration[concentration['methodology'] == 'FCS']
first, last = era.iloc[0], era.iloc[-1]
print(f"{first.year:.0f}: {first.d1_share:.1f}% of the global D1 vs {first.world_share:.1f}% of world population ({first.gap_ratio:.2f}x)")
print(f"{last.year:.0f}: {last.d1_share:.1f}% of the global D1 vs {last.world_share:.1f}% of world population ({last.gap_ratio:.2f}x)")
crossed = era[era['d1_share'] > 50]
if not crossed.empty:
    print(f"\nThe majority of the global D1 has lived in listed countries since {crossed.iloc[0].year:.0f}.")

2019: 36.6% of the global D1 vs 9.9% of world population (3.69x)
2025: 55.8% of the global D1 vs 13.5% of world population (4.13x)

The majority of the global D1 has lived in listed countries since 2022.


## Interpretation

This is a statement about geography, not about households. It says where the poorest decile lives, and makes no claim about fragility causing poverty or the reverse.

The level jump at FY20 is partly definitional, since the current FCS methodology classifies more and larger countries than the harmonized list it replaced. Only the FCS-era trend should be quoted as a change over time.